In [ ]:
# Example 1: distribution version + native _lib presence + runtime enable state.
import importlib.metadata as im
import nvtx

version = im.version("nvtx")
print(f"nvtx (NVIDIA NVTX python bindings) version = {version}")
assert isinstance(version, str) and version, "nvtx must report a distribution version"

# The Cython range/counter engine is a compiled extension — confirm it loaded.
assert nvtx._lib is not None
import nvtx._lib.lib as _libmod
print(f"native engine: {_libmod.__file__}")

# enabled() reflects whether an NVTX injection/profiler library is attached.
# GPU-less and un-profiled, this is a valid boolean either way (no-op ranges).
state = nvtx.enabled()
print(f"nvtx.enabled() = {state}")
assert isinstance(state, bool)

In [ ]:
# Example 2: the range-annotation surface — decorator, context manager,
# push/pop, start/end handle, and instantaneous mark. All are no-ops without a
# profiler attached, so they run identically GPU-less and must never raise.
import nvtx

@nvtx.annotate("compute", color="green")
def compute(n):
    return sum(range(n))

result = compute(1000)
assert result == 499500

with nvtx.annotate(message="scoped-block", color="red", domain="demo", category="io"):
    acc = sum(i * i for i in range(100))
assert acc == 328350

nvtx.push_range("manual", color="blue")   # nested pair
nvtx.pop_range()

handle = nvtx.start_range("async-span", color="rapids")  # process-agnostic span
nvtx.end_range(handle)

nvtx.mark("checkpoint", color="yellow")
print("range decorator / context-manager / push-pop / start-end / mark all OK")

In [ ]:
# Example 3: Domain-scoped counters — the NVTX3 counter API. Build a domain,
# derive a typed counter with semantics, and sample values. Un-profiled these
# become DummyCounter no-ops, but the full construction path is exercised.
import nvtx

domain = nvtx.get_domain("metrics")
assert isinstance(domain, nvtx.Domain) or domain.__class__.__name__ == "DummyDomain"

semantics = nvtx.CounterSemantics(
    unit="bytes", min=0,
    value_type=nvtx.CounterValueType.ABSOLUTE,
    interpolation=nvtx.CounterInterpolation.LINEAR,
)
counter = domain.get_counter("bytes processed", int, semantics=semantics)
for value in (4096, 8192, 16384):
    counter.sample(value)
print(f"sampled counter '{counter.__class__.__name__}' 3x under domain 'metrics'")

# Domain-scoped range + category/registered-string/timestamp helpers.
domain.push_range(message="load", color="green")
domain.pop_range()
_ = domain.get_category_id("io")
_ = domain.get_registered_string("hot-path")
print("domain range + category + registered-string helpers OK")

In [ ]:
# Example 4: Profile automatic annotation, numpy payload attachment, and the
# color helper. Profile().enable() hooks the interpreter's profile trace; with
# no injection library attached it is a safe no-op we can enable/disable around
# real work.
import numpy as np
import nvtx

profiler = nvtx.Profile()
profiler.enable()
total = 0
for step in range(3):
    with nvtx.annotate(message=f"iter-{step}", domain="loop"):
        total += step
profiler.disable()
assert total == 3

# Attach a numeric payload to a range (payload separates message from data).
arr = np.arange(8, dtype=np.int64)
with nvtx.annotate(message="reduce", color="green", payload=int(arr.sum())):
    reduced = int(arr.sum())
assert reduced == 28

# nvtx.numpy_dtype + color helper resolve without a profiler/GPU.
print("numpy_dtype(int64) =", nvtx.numpy_dtype(np.int64))
print("color_to_hex('blue') =", nvtx.colors.color_to_hex("blue"))

In [ ]:
import nvtx, torch

assert torch.cuda.is_available(), (
    f"requires-gpu cell ran but torch.cuda.is_available() is False — torch {torch.__version__}, "
    f"built against CUDA {torch.version.cuda}, device_count={torch.cuda.device_count()}"
)

domain = nvtx.get_domain("cuda-demo")
n = torch.cuda.device_count()
print(f"CUDA devices: {n} -> {[torch.cuda.get_device_name(i) for i in range(n)]}")
span = domain.start_range(message="gpu-matmul", color="green")
prod = (a := torch.randn(256, 256, device="cuda")) @ a
torch.cuda.synchronize()
domain.end_range(span)
assert torch.isfinite(prod).all().item(), "GPU matmul produced non-finite values"
print(f"GPU matmul OK under nvtx range, sum={prod.sum().item():.3f}")